# Thesis Results Dashboard

This notebook is for inspecting and plotting experiment results. Run long experiments from the terminal; this notebook reads the CSV files in `results/`, regenerates figures, and displays per-algorithm and consolidated comparison plots.

Terminal runs remain the source of truth. The notebook is just the overview layer.

## Setup

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "results":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_ROOT = PROJECT_ROOT / "results"
preferred_python = Path("/Users/Max/miniforge3/envs/streamrl_arm/bin/python")
PYTHON_BIN = str(preferred_python if preferred_python.exists() else Path(sys.executable))

SEEDS = [1, 2, 3]
TOTAL_STEPS = 1_000_000

ENVIRONMENTS = [
    {"backend": "mujoco", "env_name": "HalfCheetah-v4"},
    {"backend": "mujoco", "env_name": "Hopper-v4"},
    {"backend": "dmcontrol", "env_name": "finger-spin-v0"},
    {"backend": "dmcontrol", "env_name": "cheetah-run-v0"},
]

ALGORITHMS = {
    "td3": "TD3",
    "ac": "Stream AC(lambda)",
    "avg_A": "AVG A",
    "avg_B": "AVG B",
    "avg_C": "AVG C",
    "avg_A_ln": "AVG A + LN",
    "avg_B_ln": "AVG B + LN",
    "avg_C_ln": "AVG C + LN",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PYTHON_BIN:", PYTHON_BIN)

## Result Status

This table shows which runs already exist and the latest evaluation checkpoint for each seed.

In [ ]:
def sanitize_name(name: str) -> str:
    return name.replace("/", "__").replace(":", "_")


def run_dir(algo: str, backend: str, env_name: str, seed: int) -> Path:
    return RESULTS_ROOT / backend / algo / sanitize_name(env_name) / f"seed_{seed}"


def run_status(algo: str, backend: str, env_name: str, seed: int) -> dict:
    root = run_dir(algo, backend, env_name, seed)
    eval_csv = root / "eval.csv"
    train_csv = root / "train.csv"
    config_json = root / "config.json"
    row = {
        "algo": algo,
        "label": ALGORITHMS.get(algo, algo),
        "backend": backend,
        "env_name": env_name,
        "seed": seed,
        "exists": root.exists(),
        "eval_points": 0,
        "train_episodes": 0,
        "last_eval_step": None,
        "last_eval_mean": None,
        "last_eval_std_within_seed": None,
        "complete": False,
    }
    if config_json.exists():
        with open(config_json) as f:
            cfg = json.load(f)
        row["configured_steps"] = cfg.get("total_timesteps", cfg.get("total_steps"))
    else:
        row["configured_steps"] = None

    if eval_csv.exists() and eval_csv.stat().st_size > 0:
        df = pd.read_csv(eval_csv)
        if len(df):
            row["eval_points"] = len(df)
            row["last_eval_step"] = int(df["step"].iloc[-1])
            row["last_eval_mean"] = float(df["eval_return_mean"].iloc[-1])
            row["last_eval_std_within_seed"] = float(df["eval_return_std"].iloc[-1])
            row["complete"] = row["last_eval_step"] >= TOTAL_STEPS

    if train_csv.exists() and train_csv.stat().st_size > 0:
        row["train_episodes"] = len(pd.read_csv(train_csv))

    return row


def status_table(algorithms=ALGORITHMS.keys(), environments=ENVIRONMENTS, seeds=SEEDS) -> pd.DataFrame:
    rows = []
    for env in environments:
        for algo in algorithms:
            for seed in seeds:
                rows.append(run_status(algo, env["backend"], env["env_name"], seed))
    return pd.DataFrame(rows)

status = status_table()
status[status["exists"]].sort_values(["backend", "env_name", "algo", "seed"])

## Terminal Commands

Use these commands in a terminal for missing runs. The notebook only displays them; it does not launch jobs.

In [ ]:
def terminal_command(algo: str, backend: str, env_name: str, seed: int) -> str:
    if algo == "td3":
        return (
            f"{PYTHON_BIN} -m algorithms.td3_baseline "
            f"--backend {backend} --env-name {env_name} --seed {seed} "
            f"--total-timesteps {TOTAL_STEPS} --algo td3"
        )
    if algo == "ac":
        return (
            f"{PYTHON_BIN} -m algorithms.stream_ac_continuous "
            f"--backend {backend} --env_name {env_name} --seed {seed} "
            f"--total_steps {TOTAL_STEPS} --results_root results"
        )
    avg_modes = {
        "avg_A": ("td_only", False),
        "avg_B": ("td_reward", False),
        "avg_C": ("td_reward_entropy", False),
        "avg_A_ln": ("td_only", True),
        "avg_B_ln": ("td_reward", True),
        "avg_C_ln": ("td_reward_entropy", True),
    }
    if algo in avg_modes:
        mode, use_ln = avg_modes[algo]
        cmd = (
            f"{PYTHON_BIN} -m algorithms.avg_baseline "
            f"--backend {backend} --env_name {env_name} --seed {seed} "
            f"--total_steps {TOTAL_STEPS} --scaling_mode {mode} --results_root results"
        )
        if use_ln:
            cmd += " --use_layer_norm"
        return cmd
    raise ValueError(algo)

missing = status[~status["complete"] & status["algo"].isin(["td3", "ac", "avg_A", "avg_B", "avg_C"])]
commands = []
for _, row in missing.iterrows():
    commands.append({
        "algo": row["algo"],
        "backend": row["backend"],
        "env_name": row["env_name"],
        "seed": row["seed"],
        "command": terminal_command(row["algo"], row["backend"], row["env_name"], row["seed"]),
    })

pd.DataFrame(commands).head(20)

## Regenerate Available Plots

This reruns the existing plotters for result folders that already contain at least one `eval.csv`. With only one seed, cross-seed confidence intervals are zero; once seeds 1-3 exist, the same cells aggregate across seeds.

In [ ]:
def has_any_eval(algo: str, backend: str, env_name: str) -> bool:
    env_root = RESULTS_ROOT / backend / algo / sanitize_name(env_name)
    return any(env_root.glob("seed_*/eval.csv")) if env_root.exists() else False


def run_plotter(algo: str, backend: str, env_name: str):
    cmd = [
        PYTHON_BIN, "-m", "evaluation.fixed_plot_results",
        "--env_name", env_name,
        "--backend", backend,
        "--algo", algo,
        "--results_root", "results",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

for env in ENVIRONMENTS:
    for algo in ALGORITHMS:
        if has_any_eval(algo, env["backend"], env["env_name"]):
            run_plotter(algo, env["backend"], env["env_name"])

## Per-Algorithm Plots

In [ ]:
def figure_paths(algo: str, backend: str, env_name: str):
    fig_dir = RESULTS_ROOT / backend / algo / "figures"
    safe_env = sanitize_name(env_name)
    return fig_dir / f"{safe_env}_eval_plot.png", fig_dir / f"{safe_env}_train_plot.png"


def show_image(path: Path, width: int = 760):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"Missing: `{path.relative_to(PROJECT_ROOT)}`"))

for env in ENVIRONMENTS:
    for algo, label in ALGORITHMS.items():
        eval_png, train_png = figure_paths(algo, env["backend"], env["env_name"])
        if eval_png.exists() or train_png.exists():
            display(Markdown(f"### {label} — {env['backend']} / {env['env_name']}"))
            show_image(eval_png)
            show_image(train_png)

## Consolidated Comparison Plots

These are generated only when at least two algorithms have results for the same environment.

In [ ]:
def available_variants(backend: str, env_name: str, algorithms=ALGORITHMS) -> list[str]:
    variants = []
    for algo, label in algorithms.items():
        if has_any_eval(algo, backend, env_name):
            variants.append(f"{algo}={label}")
    return variants


def run_comparison(backend: str, env_name: str, variants: list[str], output_tag: str):
    cmd = [
        PYTHON_BIN, "-m", "evaluation.compare_variants_plotter",
        "--env_name", env_name,
        "--backend", backend,
        "--variants", *variants,
        "--results_root", "results",
        "--output_tag", output_tag,
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

comparison_outputs = []
for env in ENVIRONMENTS:
    variants = available_variants(env["backend"], env["env_name"])
    if len(variants) >= 2:
        tag = "consolidated"
        run_comparison(env["backend"], env["env_name"], variants, tag)
        safe_env = sanitize_name(env["env_name"])
        compare_dir = RESULTS_ROOT / env["backend"] / "comparisons"
        comparison_outputs.append({
            "backend": env["backend"],
            "env_name": env["env_name"],
            "eval_plot": compare_dir / f"{safe_env}_{tag}_eval.png",
            "train_plot": compare_dir / f"{safe_env}_{tag}_train.png",
            "summary": compare_dir / f"{safe_env}_{tag}_summary.csv",
        })

comparison_outputs

## Display Consolidated Plots

In [ ]:
for out in comparison_outputs:
    display(Markdown(f"### Consolidated — {out['backend']} / {out['env_name']}"))
    show_image(out["eval_plot"])
    show_image(out["train_plot"])
    if out["summary"].exists():
        display(pd.read_csv(out["summary"]))

## Final Scores From Existing Runs

In [ ]:
final_scores = status_table()
final_scores = final_scores[final_scores["exists"]].copy()
final_scores.sort_values(["backend", "env_name", "algo", "seed"])[[
    "backend", "env_name", "algo", "seed", "complete", "last_eval_step", "last_eval_mean", "last_eval_std_within_seed", "train_episodes"
]]